# Iraq Farms Health — farmland coverage check (read-only, free)
How much of Iraq's farmland (FTW field-confidence map, the same 500 m mask the planner used) falls in **no** WorldCereal tile? The planner skips tiles with too little farmland and splits Iraq at the UTM zone seams (42°E, 48°E), so some farmland can end up in no tile and is never mapped.

Reads the main run's plan in Drive and the public FTW map; writes `coverage_gaps.json`, `gaps_by_governorate.csv` and `gaps_map.png` to `iraq_winter2026/coverage/` in Drive. Submits nothing, uses no credits. Takes a few minutes. Run the cells in order and send me the printed output and the map.


In [ ]:
RUN_NAME = "iraq_winter2026"
FARMLAND_CONF = 0.22          # same threshold as the planner

from google.colab import drive
drive.mount("/content/drive")
import os
RUN = f"/content/drive/MyDrive/iraq-farms-health/worldcereal/{RUN_NAME}"
OUT = f"{RUN}/coverage"
print("Plan found:", os.path.exists(f"{RUN}/plan/grid_all.geojson"))
!pip -q install rasterio

In [ ]:
%%writefile /content/coverage_gaps.py
"""Farmland coverage check: how much FTW farmland is in no planned tile? (read-only)

python coverage_gaps.py <main_run_folder> <out_folder> [--conf 0.22]

Reads the main run's plan/grid_all.geojson (every tile, all runs share its shapes) and the FTW
field-confidence COG (the same 500 m farmland mask the planner used). Counts farmland (confidence
>= --conf, inside Iraq) that falls in no tile, by governorate (geoBoundaries ADM1), and how much of it
lies within 10 km of the UTM zone seams at 42°E and 48°E. Writes coverage_gaps.json, gaps_by_governorate.csv
and gaps_map.png. Nothing is submitted, nothing in the run folder is changed.
"""
import argparse, json, os, urllib.request
import numpy as np, rasterio, pyproj
from rasterio.features import geometry_mask, rasterize
from rasterio.windows import from_bounds
from shapely.geometry import shape, box, mapping
from shapely.ops import unary_union, transform as shp_transform
import warnings; warnings.filterwarnings("ignore")
ap = argparse.ArgumentParser(); ap.add_argument("run"); ap.add_argument("out"); ap.add_argument("--conf", type=float, default=0.22)
ap.add_argument("--ftw", help="local farmland COG (testing)"); ap.add_argument("--adm1", help="local ADM1 geojson (testing)")
A = ap.parse_args(); os.makedirs(A.out, exist_ok=True)
FTW = "/vsicurl/https://data.source.coop/ftw/global-data/predictions/confidence/confidence/prue_v1_confidence_global.tif"
os.environ.setdefault("GDAL_DISABLE_READDIR_ON_OPEN", "EMPTY_DIR"); os.environ.setdefault("CPL_VSIL_CURL_ALLOWED_EXTENSIONS", ".tif")

def gb(level):
    try:
        meta = json.load(urllib.request.urlopen(f"https://www.geoboundaries.org/api/current/gbOpen/IRQ/{level}/", timeout=60))
        url = meta["gjDownloadURL"]
    except Exception:
        url = f"https://github.com/wmgeolab/geoBoundaries/raw/main/releaseData/gbOpen/IRQ/{level}/geoBoundaries-IRQ-{level}.geojson"
    return json.load(urllib.request.urlopen(url, timeout=120))
adm1 = json.load(open(A.adm1)) if A.adm1 else gb("ADM1")
govs = [(f["properties"]["shapeName"], shape(f["geometry"]).buffer(0)) for f in adm1["features"]]
iraq = unary_union([g for _, g in govs])
plan = json.load(open(os.path.join(A.run, "plan", "grid_all.geojson")))
tiles = unary_union([shape(f["geometry"]) for f in plan["features"]])
W, S, E, N = iraq.bounds
with rasterio.open(A.ftw or FTW) as ds:
    to_r = pyproj.Transformer.from_crs(4326, ds.crs, always_xy=True).transform
    bw, bs, be, bn = shp_transform(to_r, box(W, S, E, N)).bounds
    win = from_bounds(bw, bs, be, bn, ds.transform).round_offsets().round_lengths()
    conf = ds.read(1, window=win, masked=True).filled(-1).astype("float32")
    tr = ds.window_transform(win); crs = ds.crs
    if ds.nodata is not None: conf[conf == ds.nodata] = -1
def mask(geom):
    return ~geometry_mask([mapping(shp_transform(to_r, geom))], conf.shape, tr, invert=False) if not geom.is_empty else np.zeros(conf.shape, bool)
in_iraq = mask(iraq); in_tile = mask(tiles)
farm = (conf >= A.conf) & in_iraq
# pixel area in km² by row (geographic or projected source)
if crs.is_geographic:
    lat = tr.f + (np.arange(conf.shape[0]) + 0.5) * tr.e
    px_km2 = (abs(tr.a) * 111.32 * np.cos(np.radians(lat))) * (abs(tr.e) * 110.57)
    AREA = np.broadcast_to(px_km2[:, None], conf.shape)
else:
    AREA = np.full(conf.shape, abs(tr.a * tr.e) / 1e6)
gap = farm & ~in_tile
inv = pyproj.Transformer.from_crs(crs, 4326, always_xy=True).transform
cols_x = tr.c + (np.arange(conf.shape[1]) + 0.5) * tr.a
lon_row = np.array(inv(cols_x, np.full_like(cols_x, (tr.f + tr.e * conf.shape[0] / 2)))[0])
LON = np.broadcast_to(lon_row[None, :], conf.shape)
seam = (np.abs(LON - 42) <= 0.1) | (np.abs(LON - 48) <= 0.1)
res = {"conf": A.conf, "farmland_km2": float(AREA[farm].sum()), "covered_km2": float(AREA[farm & in_tile].sum()),
       "gap_km2": float(AREA[gap].sum()), "gap_near_42E_48E_km2": float(AREA[gap & seam].sum()),
       "farmland_near_42E_48E_km2": float(AREA[farm & seam].sum()), "governorates": []}
for name, g in govs:
    m = mask(g)
    f_ = float(AREA[farm & m].sum()); gp = float(AREA[gap & m].sum())
    res["governorates"].append({"name": name, "farmland_km2": round(f_, 1), "gap_km2": round(gp, 1), "gap_share": round(gp / f_, 3) if f_ else None})
res["governorates"].sort(key=lambda r: -r["gap_km2"])
json.dump(res, open(os.path.join(A.out, "coverage_gaps.json"), "w"), indent=1)
with open(os.path.join(A.out, "gaps_by_governorate.csv"), "w") as fh:
    fh.write("governorate,farmland_km2,gap_km2,gap_share\n")
    for r in res["governorates"]: fh.write(f"{r['name']},{r['farmland_km2']},{r['gap_km2']},{r['gap_share']}\n")
print(f"FTW farmland (conf ≥ {A.conf}) in Iraq: {res['farmland_km2']:,.0f} km²")
print(f"  inside planned tiles:  {res['covered_km2']:,.0f} km² ({res['covered_km2']/res['farmland_km2']:.1%})")
print(f"  in no tile:            {res['gap_km2']:,.0f} km² ({res['gap_km2']/res['farmland_km2']:.1%})")
print(f"  of which within ~10 km of 42°E/48°E: {res['gap_near_42E_48E_km2']:,.0f} km² "
      f"(farmland there: {res['farmland_near_42E_48E_km2']:,.0f} km²)")
print("\nBy governorate (largest gaps first):")
for r in res["governorates"]:
    print(f"  {r['name']:<22} farmland {r['farmland_km2']:>9,.0f} km²   in no tile {r['gap_km2']:>8,.0f} km²"
          + (f"  ({r['gap_share']:.1%})" if r["gap_share"] is not None else ""))
try:
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    img = np.zeros(conf.shape + (3,), np.uint8); img[:] = 235
    img[farm & in_tile] = (127, 168, 106); img[gap] = (200, 40, 40)
    fig, ax = plt.subplots(figsize=(8, 8), dpi=130); ax.imshow(img); ax.set_axis_off()
    ax.set_title(f"FTW farmland: green = in a tile, red = in no tile ({res['gap_km2']:,.0f} km²)")
    fig.savefig(os.path.join(A.out, "gaps_map.png"), bbox_inches="tight"); print("\nMap:", os.path.join(A.out, "gaps_map.png"))
except Exception as e:
    print("(map skipped:", e, ")")


In [ ]:
!python /content/coverage_gaps.py "$RUN" "$OUT" --conf $FARMLAND_CONF
from IPython.display import Image, display
if os.path.exists(f"{OUT}/gaps_map.png"):
    display(Image(f"{OUT}/gaps_map.png", width=700))